In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [2]:
df = pd.read_csv("diabetes_type2.csv")
df

,Patient_ID,Age,Gender,BMI,Glucose_Level,Insulin,Blood_Pressure,Skin_Thickness,Diabetes_Pedigree_Function,HbA1c,Cholesterol,Medication,Outcome
0,P1000,old,Female,NaN,134,192,twenty,44,2.445,9.52,twenty,Metformin,1
1,P1001,42,Female,34.2,145,17,83,37,1.305,7.2,error,Metformin,1
2,P1002,69,Male,37,170,NaN,138,42,2.265,11.6,300,NaN,1
3,P1003,49,Male,25.4,70,146,NaN,NaN,0.652,11.29,173,NaN,1
4,P1004,22,Male,unknown,NaN,194,103,31,0.106,8.58,241,Insulin,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3995,P4995,46,Female,36.4,160,85,119,43,1.605,11.97,135,NaN,1
3996,P4996,63,Male,32.6,194,205,121,18,1.279,NaN,NaN,Metformin,1
3997,P4997,57,Female,22,155,104,136,49,0.362,NaN,233,Insulin,1
3998,3,38,Female,35.3,99,200,88,12,1.909,10.42,292,Insulin,1


In [6]:
print("ORIGINAL DATASET INFO")
df.shape

ORIGINAL DATASET INFO


(4000, 13)

In [7]:
df.columns.tolist()

['Patient_ID',
 'Age',
 'Gender',
 'BMI',
 'Glucose_Level',
 'Insulin',
 'Blood_Pressure',
 'Skin_Thickness',
 'Diabetes_Pedigree_Function',
 'HbA1c',
 'Cholesterol',
 'Medication',
 'Outcome']

In [8]:
df.dtypes

Patient_ID                    object
Age                           object
Gender                        object
BMI                           object
Glucose_Level                 object
Insulin                       object
Blood_Pressure                object
Skin_Thickness                object
Diabetes_Pedigree_Function    object
HbA1c                         object
Cholesterol                   object
Medication                    object
Outcome                        int64
dtype: object

In [9]:
df.isnull().sum()

Patient_ID                     261
Age                            330
Gender                         299
BMI                            301
Glucose_Level                  314
Insulin                        317
Blood_Pressure                 372
Skin_Thickness                 318
Diabetes_Pedigree_Function     320
HbA1c                          360
Cholesterol                    307
Medication                    1449
Outcome                          0
dtype: int64

In [10]:
df.duplicated().sum()

np.int64(0)

In [11]:
df['Patient_ID'].duplicated(keep=False).sum()

np.int64(451)

In [12]:
df.drop(columns=["Patient_ID"], inplace=True)
print("Dropped 'Patient_ID' column.")

Dropped 'Patient_ID' column.


In [13]:
JUNK = ["old", "error", "twenty", "unknown"]
NUMERIC_COLS = [
    "Age", "BMI", "Glucose_Level", "Insulin",
    "Blood_Pressure", "Skin_Thickness",
    "Diabetes_Pedigree_Function", "HbA1c", "Cholesterol"
]
for col in NUMERIC_COLS:
    before = df[col].isin(JUNK).sum()
    df[col] = df[col].replace(JUNK, np.nan)
    print(f"[2] {col}: replaced {before} junk strings with NaN.")


[2] Age: replaced 166 junk strings with NaN.
[2] BMI: replaced 177 junk strings with NaN.
[2] Glucose_Level: replaced 149 junk strings with NaN.
[2] Insulin: replaced 159 junk strings with NaN.
[2] Blood_Pressure: replaced 156 junk strings with NaN.
[2] Skin_Thickness: replaced 173 junk strings with NaN.
[2] Diabetes_Pedigree_Function: replaced 147 junk strings with NaN.
[2] HbA1c: replaced 161 junk strings with NaN.
[2] Cholesterol: replaced 157 junk strings with NaN.


In [15]:
for col in NUMERIC_COLS:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("All numeric columns cast to float64")
print(f"    Dtypes after cast:\n{df[NUMERIC_COLS].dtypes.to_string()}")

All numeric columns cast to float64
    Dtypes after cast:
Age                           float64
BMI                           float64
Glucose_Level                 float64
Insulin                       float64
Blood_Pressure                float64
Skin_Thickness                float64
Diabetes_Pedigree_Function    float64
HbA1c                         float64
Cholesterol                   float64


In [16]:
bad_gender_mask = ~df["Gender"].isin(["Male", "Female"]) & df["Gender"].notna()
bad_gender_count = bad_gender_mask.sum()
df.loc[bad_gender_mask, "Gender"] = np.nan
print(f"\n[4] Gender: set {bad_gender_count} invalid entries to NaN.")

gender_mode = df["Gender"].mode()[0]
df["Gender"].fillna(gender_mode, inplace=True)
print(f"    Gender: filled NaN with mode='{gender_mode}'. NaN remaining: {df['Gender'].isnull().sum()}")



[4] Gender: set 210 invalid entries to NaN.
    Gender: filled NaN with mode='Female'. NaN remaining: 0


C:\Users\DEEPANSHI\AppData\Local\Temp\ipykernel_3032\1346354171.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["Gender"].fillna(gender_mode, inplace=True)


In [17]:
med_numeric_mask = pd.to_numeric(df["Medication"], errors="coerce").notna()
bad_med_count = med_numeric_mask.sum()
df.loc[med_numeric_mask, "Medication"] = np.nan
print(f"\n[5] Medication: set {bad_med_count} numeric-string entries to NaN.")

df["Medication"].fillna("None", inplace=True)
print(f"    Medication: filled NaN with 'None'. NaN remaining: {df['Medication'].isnull().sum()}")
print(f"    Medication unique values: {df['Medication'].unique()}")



[5] Medication: set 168 numeric-string entries to NaN.
    Medication: filled NaN with 'None'. NaN remaining: 0
    Medication unique values: ['Metformin' 'None' 'Insulin']


C:\Users\DEEPANSHI\AppData\Local\Temp\ipykernel_3032\237121664.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["Medication"].fillna("None", inplace=True)


In [18]:
for col in NUMERIC_COLS:
    null_before = df[col].isnull().sum()
    if null_before > 0:
        med = df[col].median()
        df[col].fillna(med, inplace=True)
        print(f"    {col}: filled {null_before} NaN → median={med:.3f}")


    Age: filled 496 NaN → median=49.000
    BMI: filled 478 NaN → median=29.100
    Glucose_Level: filled 463 NaN → median=134.000
    Insulin: filled 476 NaN → median=146.000
    Blood_Pressure: filled 528 NaN → median=99.500
    Skin_Thickness: filled 491 NaN → median=30.000
    Diabetes_Pedigree_Function: filled 467 NaN → median=1.326
    HbA1c: filled 521 NaN → median=8.500
    Cholesterol: filled 464 NaN → median=210.000


C:\Users\DEEPANSHI\AppData\Local\Temp\ipykernel_3032\4035913144.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(med, inplace=True)


In [20]:
remaining_nulls = df.isnull().sum().sum()
print(f" Total NaN remaining after cleaning: {remaining_nulls}")

 Total NaN remaining after cleaning: 0


In [21]:
le_gender = LabelEncoder()
le_medication = LabelEncoder()

df["Gender"] = le_gender.fit_transform(df["Gender"])
df["Medication"] = le_medication.fit_transform(df["Medication"])

print("Label Encoding done.")
print(f"    Gender classes     : {list(le_gender.classes_)}  → {list(range(len(le_gender.classes_)))}")
print(f"    Medication classes : {list(le_medication.classes_)}  → {list(range(len(le_medication.classes_)))}")


Label Encoding done.
    Gender classes     : ['Female', 'Male']  → [0, 1]
    Medication classes : ['Insulin', 'Metformin', 'None']  → [0, 1, 2]


In [23]:
X = df.drop(columns=["Outcome"])
y = df["Outcome"]

print(f" Features (X) shape : {X.shape}")
print(f"    Target  (y) shape  : {y.shape}")
print(f"    Feature columns    : {X.columns.tolist()}")

 Features (X) shape : (4000, 11)
    Target  (y) shape  : (4000,)
    Feature columns    : ['Age', 'Gender', 'BMI', 'Glucose_Level', 'Insulin', 'Blood_Pressure', 'Skin_Thickness', 'Diabetes_Pedigree_Function', 'HbA1c', 'Cholesterol', 'Medication']


In [24]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print("\n[10] StandardScaler applied.")
print(f"     Mean of scaled features (should be ~0):\n{X_scaled.mean().round(6).to_string()}")
print(f"\n     Std  of scaled features (should be ~1):\n{X_scaled.std().round(6).to_string()}")


[10] StandardScaler applied.
     Mean of scaled features (should be ~0):
Age                           0.0
Gender                       -0.0
BMI                           0.0
Glucose_Level                 0.0
Insulin                      -0.0
Blood_Pressure               -0.0
Skin_Thickness               -0.0
Diabetes_Pedigree_Function   -0.0
HbA1c                        -0.0
Cholesterol                  -0.0
Medication                   -0.0

     Std  of scaled features (should be ~1):
Age                           1.000125
Gender                        1.000125
BMI                           1.000125
Glucose_Level                 1.000125
Insulin                       1.000125
Blood_Pressure                1.000125
Skin_Thickness                1.000125
Diabetes_Pedigree_Function    1.000125
HbA1c                         1.000125
Cholesterol                   1.000125
Medication                    1.000125


In [25]:
df_final = X_scaled.copy()
df_final["Outcome"] = y.values
df_final.to_csv("diabetes_cleaneddataset.csv", index=False)

In [26]:
df_final.shape

(4000, 12)